In [25]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from bs4 import BeautifulSoup

from time import sleep

from datetime import datetime

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import datetime

import os

from selenium.webdriver.chrome.service import Service as ChromeService

# %%

In [ ]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BA BARS'

print(f"Running{regulatorName} Web Scraping Tool v.1.2")


now=datetime.datetime.now()

filename= 'BA BARS Data {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


RunningBA BARS Web Scraping Tool v.1.2


In [27]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

In [28]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="ccc-notify-accept"]/span[contains(text(),"Accept")]').click()

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)
        

# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")



    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Wait to load the page

        sleep(3)
        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()
    

def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="cookiesjsr"]/div/div/div/div/div[2]/button[2]').click()
        print('[Success] : Success to Click Cookie')

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)
        

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(1)
                break
            except:
                print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(1)
                if key_press:                    
                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')


In [29]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={'BA BARS 1': 'https://abrs.ba/en/category/institutions/banks/inst-banks-based-in-rs/', 
         'BA BARS 2': 'https://abrs.ba/en/category/institutions/microcredit-organizations/inst-mco-based-in-rs/'}


Typology={'BA BARS 1': 'Banks based in Republika Srpska',

        'BA BARS 2': 'MCO based in Republika Srpska'}



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
		  

In [30]:
# %%
#------------------------------------------------ Begin_Main ----------------------------------------

driver.delete_all_cookies()
for reg in regdict:
	inner_links = []
	next_page = 1
	print('Working with {}.'.format(reg))
	driver.get(regdict[reg])

	while next_page:
		
		sleep(1)
     
		soup=BeautifulSoup(driver.page_source, 'html.parser')
		
		try:
			blocks=soup.find("div", {"et_pb_ajax_pagination_container"}).find_all("article")
			next_ = soup.find('div',{'wp-pagenavi'}).find('a',{'page larger'})
			
		except:
			print('Only one page')
			next_page = 0
   
		for block in blocks:
			name_homepage = block.find('h2').text
			#print(name_homepage)
			sleep(2)
			inner_link = block.find('a')['href']
			inner_links.append(inner_link)

		
		if next_page and next_ is not None:
			#next_content = next_['href']
			driver.get(next_['href'])
			#driver.get(next_content)
			print('-----Next Page')
			#print(next_)
			continue
		else:
			pass

		for inner_link in inner_links:
			driver.get(inner_link)
			sleep(1)
			soup2 = BeautifulSoup(driver.page_source, 'html.parser')
			name = soup2.find('h1').text
			#print(name)
			sqldict['ListProcessDate'].append(processdate)
			contents = soup2.find('div',{'et_pb_module et_pb_post_content et_pb_post_content_0_tb_body'})
			# print(len(contents))
			sqldict['Name'].append(name.encode('utf-8'))
			sqldict['RegCtry'].append(reg.split()[0])
			sqldict['RegCode'].append(reg.split()[1])
			sqldict['ListCode'].append(reg.split()[2])
			sqldict['ListName'].append(Typology[reg])
			sqldict['RegulationType'].append('Regulated')
			for p in contents.find_all('p'):
				#print(p.text)
				if p.text!='':
					if p.text.split(':')[0] == 'Address'and len(sqldict['ListProcessDate'])>len(sqldict['Address_1']):
						sqldict['Address_1'].append(p.text.split(':')[-1])
						#sqldict = bourange_same_length_array(sqldict) 
					elif p.text.split(':')[0] == 'Phone'and len(sqldict['ListProcessDate'])>len(sqldict['Phone']):
						sqldict['Phone'].append(p.text.split(':')[-1])
						#sqldict = bourange_same_length_array(sqldict) 
					elif p.text.split(':')[0] == 'Fax'and len(sqldict['ListProcessDate'])>len(sqldict['Fax']):
						sqldict['Fax'].append(p.text.split(':')[-1])
						#sqldict = bourange_same_length_array(sqldict) 
					elif (p.text.split(':')[0] == 'E-pošta' or p.text.split(':')[0] == 'E-mail') and len(sqldict['ListProcessDate'])>len(sqldict['Email']):
						sqldict['Email'].append(p.text.split(':')[-1])
						#sqldict = bourange_same_length_array(sqldict) 
					elif p.text.split(':')[0] == 'WEB' and len(sqldict['ListProcessDate'])>len(sqldict['Website']):
						sqldict['Website'].append(p.text.split(':')[-1])  
			sqldict = bourange_same_length_array(sqldict) 
	
sqldict = bourange_same_length_array(sqldict) 

Working with BA BARS 1.
Only one page
Working with BA BARS 2.
-----Next Page
Only one page


In [31]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\6\ipykernel_22888\3068940208.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [32]:
df.to_csv('TOTAL7.csv')

In [ ]:
df.to_csv('total.csv')